In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

#### Q1. How many lesson pages

**Answer: 72** (see next cell output)

In [3]:
len(documents)

72

In [4]:
documents[:2]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

#### Q2. Indexing and searching

Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?


In [5]:
from minsearch import Index


index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query)

In [6]:
results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

In [13]:
from rag_helper2 import RAGBase
from openai import OpenAI

index = Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)

index.fit(documents)

client = OpenAI()

rag = RAGBase(index=index, llm_client=client)

question = "How does the agentic loop keep calling the model until it stops??"
answer, usage = rag.rag(question)

print(answer)
print(usage.input_tokens)

It keeps calling the model with a `while True` loop.

Each time:
1. Send the full `messages` history to the model.
2. Check the response for any `function_call` items.
3. If there are tool calls, run them and append the results to `messages`.
4. If there are no function calls, `break` out of the loop.

So the stop condition is: **no function calls in the model’s response**.

In the lesson code, that’s this part:

```python
has_function_calls = False
...
if item.type == "function_call":
    ...
    has_function_calls = True
...
if has_function_calls == False:
    break
```

So the agentic loop doesn’t “know” in advance how many times to call the model. The model decides whether to ask for more tool use, and the loop keeps going until it returns a final answer without tool calls.
7126


it was a little over 7000 input tokens


In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
len(chunks)

295

In [ ]:
chunk_index = Index(
    text_fields=["contents"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

rag = RAGBase(index=chunk_index, llm_client=client)

question = "How does the agentic loop keep calling the model until it stops?"
answer, usage = rag.rag(question)

print(answer)
print("Input tokens:", usage.input_tokens)

I don't know.
Input tokens: 77


In [14]:
from minsearch import Index
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [15]:
from typing import List, Dict, Any

class SearchTools:
    def __init__(self, index):
        self.index = index
        self.search_calls = 0

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the course materials for chunks relevant to the student's question.

        Args:
            query (str): Search query text.

        Returns:
            List[Dict[str, Any]]: A list of matching document chunks.
        """
        self.search_calls += 1
        return self.index.search(query=query, num_results=5)

In [18]:
from minsearch import Index
from gitsource import chunk_documents
from typing import List, Dict, Any
from openai import OpenAI

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)


class SearchTools:
    def __init__(self, index):
        self.index = index
        self.search_calls = 0

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the course chunks for relevant text.
        """
        self.search_calls += 1
        return self.index.search(query=query, num_results=5)


search_tools = SearchTools(chunk_index)
client = OpenAI()

question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    {"role": "developer", "content": """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()},
    {"role": "user", "content": question}
]

# Simple agentic loop
while True:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=messages
    )

    if not response.tools or not response.output:
        # No tool call, just an answer
        print(response.output_text)
        break

    # If there's a tool call, execute it
    for part in response.output:
        if part.type == "tool_call":
            tool_name = part.name
            if tool_name == "search":
                args = part.arguments
                query = args["query"]
                result = search_tools.search(query)
                messages.append({
                    "role": "tool",
                    "tool_call_id": part.tool_call_id,
                    "content": str(result)
                })
            else:
                messages.append({
                    "role": "tool",
                    "tool_call_id": part.tool_call_id,
                    "content": "Unknown tool"
                })

print("Search calls:", search_tools.search_calls)

Searching for sources on agentic loop and RAG differences.Searching with broader terms: agentic loop, tool use, retrieval augmented generation differences.An **agentic loop** is a repeated cycle where the model:

1. **Observes** the current state or user request,
2. **Plans** the next step,
3. **Acts** by calling a tool, retrieving info, or taking another operation,
4. **Reads the result**,
5. **Repeats** until it decides the task is done.

A simple way to think about it is:

**reason → act → observe → reason → act → …**

## How it works
In an agentic system, the model is not just answering once. It can:
- decide what to do next,
- choose among multiple tools,
- refine its plan based on intermediate results,
- stop when it has enough evidence or has completed the task.

Example:
- User asks: “Find the cheapest flight and summarize the baggage policy.”
- Agent:
  1. searches flights,
  2. checks airline baggage policy,
  3. compares options,
  4. writes a final answer.

## How that diff

In [22]:
from minsearch import Index
from gitsource import chunk_documents
from typing import List, Dict, Any
from pydantic_ai import Agent

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)


class SearchTools:
    def __init__(self, index):
        self.index = index
        self.search_calls = 0

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the course chunks for relevant text.

        Args:
            query: Search query text.

        Returns:
            A list of matching document chunks.
        """
        self.search_calls += 1
        return self.index.search(query=query, num_results=5)


search_tools = SearchTools(chunk_index)

agent = Agent(
    "openai:gpt-5.4-mini",
    system_prompt=(
        "You're a course teaching assistant. "
        "Answer the student's question using the search tool. "
        "Make multiple searches with different keywords before answering."
    )
)

@agent.tool_plain
def search(query: str) -> List[Dict[str, Any]]:
    """
    Search the course chunks for relevant text.

    Args:
        query: Search query text.

    Returns:
        A list of matching document chunks.
    """
    return search_tools.search(query)

In [25]:
question = "How does the agentic loop work, and how is it different from plain RAG?"
result = await agent.run(question)

print(result.output)
print("Search calls:", search_tools.search_calls)

The **agentic loop** is an iterative control pattern:

1. Send the user goal to the LLM
2. If the model asks to use a tool, execute that tool
3. Feed the tool result back to the model
4. Repeat until the model returns a final answer with no more tool calls

In the course notes, this is described as a `while` loop that “called the LLM, executed any tool calls it returned, sent the results back, and stopped when the model produced a final answer.” The key idea is that **the model decides what to do next** during the loop.

### How it differs from plain RAG

**Plain RAG** is usually a fixed pipeline:

```python
search_results = search(question)
prompt = build_prompt(question, search_results)
answer = llm(prompt)
```

So RAG is mainly:

- retrieve relevant context once
- stuff that context into the prompt
- generate an answer

### Main difference

- **RAG**: one retrieval step, then generation
- **Agentic loop**: multiple rounds of reasoning + tool use + feedback

### Practical implication